In [1]:
"""
Data Augmentation Script for Brain Tumor MRI Dataset
Author: Your Name
Project: 10-Class Brain Tumor Detection

This script applies data augmentation techniques to increase dataset size
and improve model generalization, following the methodology from:
Ishfaq et al. (2025) - Scientific Reports

Augmentation techniques applied:
- Rotation
- Width/Height shifting
- Zoom
- Shear transformation
- Horizontal and vertical flipping
- Brightness adjustment
"""

import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import json
from datetime import datetime
from scipy import ndimage


class DataAugmentor:
    """
    Apply data augmentation to brain MRI images
    """

    def __init__(self, input_dir, output_dir, target_count_per_class=2000):
        """
        Initialize the augmentor

        Args:
            input_dir (str): Path to preprocessed dataset
            output_dir (str): Path to save augmented images
            target_count_per_class (int): Target number of images per class after augmentation
        """
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir)
        self.target_count = target_count_per_class

        # Define the 10 classes
        self.classes = [
            'Astrocytoma',
            'Ependymoma',
            'Glioblastoma',
            'Germinoma',
            'Medulloblastoma',
            'Meningioma',
            'Oligodendroglioma',
            'Pituitary',
            'Schwannoma',
            'No_Tumor'
        ]

        self.stats = {
            'original_count': {},
            'augmented_count': {},
            'total_count': {}
        }

    def rotate_image(self, image, angle_range=(-20, 20)):
        """
        Rotate image by random angle

        Args:
            image: Input image
            angle_range: Range of rotation angles in degrees

        Returns:
            Rotated image
        """
        angle = np.random.uniform(angle_range[0], angle_range[1])
        rotated = ndimage.rotate(image, angle, reshape=False, mode='nearest')
        return rotated

    def shift_image(self, image, width_shift_range=0.1, height_shift_range=0.1):
        """
        Shift image horizontally and vertically

        Args:
            image: Input image
            width_shift_range: Fraction of width to shift
            height_shift_range: Fraction of height to shift

        Returns:
            Shifted image
        """
        h, w = image.shape[:2]

        # Calculate shift amounts
        tx = int(np.random.uniform(-width_shift_range, width_shift_range) * w)
        ty = int(np.random.uniform(-height_shift_range, height_shift_range) * h)

        # Create transformation matrix
        M = np.float32([[1, 0, tx], [0, 1, ty]])

        # Apply shift
        shifted = cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        return shifted

    def zoom_image(self, image, zoom_range=(0.9, 1.1)):
        """
        Zoom in/out on image

        Args:
            image: Input image
            zoom_range: Range of zoom factors

        Returns:
            Zoomed image
        """
        h, w = image.shape[:2]
        zoom_factor = np.random.uniform(zoom_range[0], zoom_range[1])

        # Calculate new dimensions
        new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)

        # Resize
        resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

        # Crop or pad to original size
        if zoom_factor > 1:
            # Crop center
            start_h = (new_h - h) // 2
            start_w = (new_w - w) // 2
            result = resized[start_h:start_h+h, start_w:start_w+w]
        else:
            # Pad to original size
            result = np.zeros_like(image)
            start_h = (h - new_h) // 2
            start_w = (w - new_w) // 2
            result[start_h:start_h+new_h, start_w:start_w+new_w] = resized

        return result

    def shear_image(self, image, shear_range=0.2):
        """
        Apply shear transformation to image

        Args:
            image: Input image
            shear_range: Shear intensity

        Returns:
            Sheared image
        """
        h, w = image.shape[:2]
        shear = np.random.uniform(-shear_range, shear_range)

        # Shear matrix
        M = np.float32([[1, shear, 0], [0, 1, 0]])

        sheared = cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        return sheared

    def flip_image(self, image, horizontal=True, vertical=False):
        """
        Flip image horizontally and/or vertically

        Args:
            image: Input image
            horizontal: Apply horizontal flip
            vertical: Apply vertical flip

        Returns:
            Flipped image
        """
        if horizontal and np.random.random() > 0.5:
            image = cv2.flip(image, 1)

        if vertical and np.random.random() > 0.5:
            image = cv2.flip(image, 0)

        return image

    def adjust_brightness(self, image, brightness_range=(0.8, 1.2)):
        """
        Adjust image brightness

        Args:
            image: Input image
            brightness_range: Range of brightness factors

        Returns:
            Brightness-adjusted image
        """
        factor = np.random.uniform(brightness_range[0], brightness_range[1])
        adjusted = cv2.convertScaleAbs(image, alpha=factor, beta=0)
        return adjusted

    def augment_image(self, image):
        """
        Apply random augmentation to image

        Args:
            image: Input image

        Returns:
            Augmented image
        """
        # Randomly select which augmentations to apply
        augmented = image.copy()

        # Rotation (80% chance)
        if np.random.random() > 0.2:
            augmented = self.rotate_image(augmented)

        # Shift (60% chance)
        if np.random.random() > 0.4:
            augmented = self.shift_image(augmented)

        # Zoom (50% chance)
        if np.random.random() > 0.5:
            augmented = self.zoom_image(augmented)

        # Shear (40% chance)
        if np.random.random() > 0.6:
            augmented = self.shear_image(augmented)

        # Flip (50% chance)
        augmented = self.flip_image(augmented, horizontal=True, vertical=False)

        # Brightness (70% chance)
        if np.random.random() > 0.3:
            augmented = self.adjust_brightness(augmented)

        return augmented

    def augment_class(self, class_name):
        """
        Augment images for a specific class

        Args:
            class_name (str): Name of the class to augment
        """
        class_input_dir = self.input_dir / class_name
        class_output_dir = self.output_dir / class_name

        # Create output directory
        class_output_dir.mkdir(parents=True, exist_ok=True)

        # Get all image files
        image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']
        image_files = []
        for ext in image_extensions:
            image_files.extend(list(class_input_dir.glob(f'*{ext}')))
            image_files.extend(list(class_input_dir.glob(f'*{ext.upper()}')))

        if not image_files:
            print(f"  ⚠️  No images found in {class_input_dir}")
            return

        original_count = len(image_files)
        self.stats['original_count'][class_name] = original_count

        # Copy original images
        print(f"  Copying {original_count} original images...")
        for img_path in image_files:
            output_path = class_output_dir / img_path.name
            cv2.imwrite(str(output_path), cv2.imread(str(img_path)))

        # Calculate how many augmented images needed
        if original_count >= self.target_count:
            print(f"  ✓ Already have {original_count} images (target: {self.target_count})")
            self.stats['augmented_count'][class_name] = 0
            self.stats['total_count'][class_name] = original_count
            return

        augmented_needed = self.target_count - original_count

        print(f"  Generating {augmented_needed} augmented images...")

        augmented_count = 0
        pbar = tqdm(total=augmented_needed, desc=f"  Augmenting {class_name}")

        while augmented_count < augmented_needed:
            # Select random original image
            img_path = np.random.choice(image_files)

            # Read image
            image = cv2.imread(str(img_path))

            if image is None:
                continue

            # Apply augmentation
            augmented = self.augment_image(image)

            # Save augmented image
            output_path = class_output_dir / f"{img_path.stem}_aug_{augmented_count}.png"
            cv2.imwrite(str(output_path), augmented)

            augmented_count += 1
            pbar.update(1)

        pbar.close()

        self.stats['augmented_count'][class_name] = augmented_count
        self.stats['total_count'][class_name] = original_count + augmented_count

    def augment_dataset(self):
        """
        Augment the entire dataset
        """
        print("=" * 80)
        print("🔄 DATA AUGMENTATION PIPELINE")
        print("=" * 80)
        print(f"Input directory: {self.input_dir}")
        print(f"Output directory: {self.output_dir}")
        print(f"Target images per class: {self.target_count}")
        print("=" * 80)

        for class_name in self.classes:
            print(f"\n📊 Processing class: {class_name}")

            if not (self.input_dir / class_name).exists():
                print(f"  ⚠️  Directory not found, skipping...")
                continue

            self.augment_class(class_name)

        # Print statistics
        self.print_statistics()

        # Save statistics
        self.save_statistics()

    def print_statistics(self):
        """Print augmentation statistics"""
        print("\n" + "=" * 80)
        print("📊 AUGMENTATION STATISTICS")
        print("=" * 80)

        print(f"\n{'Class':<20} {'Original':<12} {'Augmented':<12} {'Total':<12}")
        print("-" * 60)

        for class_name in self.classes:
            if class_name in self.stats['original_count']:
                original = self.stats['original_count'][class_name]
                augmented = self.stats['augmented_count'][class_name]
                total = self.stats['total_count'][class_name]
                print(f"{class_name:<20} {original:<12} {augmented:<12} {total:<12}")

        print("=" * 80)

    def save_statistics(self):
        """Save statistics to JSON file"""
        stats_file = self.output_dir / 'augmentation_stats.json'

        stats_data = {
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'input_directory': str(self.input_dir),
            'output_directory': str(self.output_dir),
            'target_count_per_class': self.target_count,
            'statistics': self.stats
        }

        with open(stats_file, 'w') as f:
            json.dump(stats_data, f, indent=2)

        print(f"\nStatistics saved to: {stats_file}")


def main():
    """
    Main execution function
    """
    # Configuration
    INPUT_DIR = "E:/Semisters/brain tumer prediction/brain_mri_10class_processed"  # Preprocessed dataset
    OUTPUT_DIR = "brain_mri_10class_augmented"  # Output directory for augmented data
    TARGET_COUNT = 2000  # Target number of images per class

    print("\n Starting Data Augmentation Pipeline\n")

    # Check if input directory exists
    if not os.path.exists(INPUT_DIR):
        print(f" Error: Input directory '{INPUT_DIR}' not found!")
        print(f"Please run preprocess_brain_mri.py first")
        return

    # Create augmentor
    augmentor = DataAugmentor(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        target_count_per_class=TARGET_COUNT
    )

    # Perform augmentation
    augmentor.augment_dataset()

    print("\n✅ Augmentation completed!")
    print(f"📂 Augmented data saved to: {OUTPUT_DIR}")
    print("\n📝 Note: Use the augmented dataset for training to improve model performance")
    print("   See the paper (Table 2) for augmentation results")


if __name__ == "__main__":
    main()


 Starting Data Augmentation Pipeline

🔄 DATA AUGMENTATION PIPELINE
Input directory: E:\Semisters\brain tumer prediction\brain_mri_10class_processed
Output directory: brain_mri_10class_augmented
Target images per class: 2000

📊 Processing class: Astrocytoma
  Copying 1128 original images...
  Generating 872 augmented images...


  Augmenting Astrocytoma: 100%|██████████| 872/872 [00:30<00:00, 28.91it/s]



📊 Processing class: Ependymoma
  Copying 300 original images...
  Generating 1700 augmented images...


  Augmenting Ependymoma: 100%|██████████| 1700/1700 [01:01<00:00, 27.42it/s]



📊 Processing class: Glioblastoma
  Copying 408 original images...
  Generating 1592 augmented images...


...


📊 Processing class: Meningioma
  Copying 5164 original images...
  ✓ Already have 5164 images (target: 2000)

📊 Processing class: Oligodendroglioma
  Copying 448 original images...
  Generating 1552 augmented images...


  Augmenting Oligodendroglioma: 100%|██████████| 1552/1552 [00:51<00:00, 30.31it/s]



📊 Processing class: Pituitary
  Copying 5316 original images...
  ✓ Already have 5316 images (target: 2000)

📊 Processing class: Schwannoma
  Copying 930 original images...
  Generating 1070 augmented images...


  Augmenting Schwannoma: 100%|██████████| 1070/1070 [00:38<00:00, 28.03it/s]



📊 Processing class: No_Tumor
  Copying 4792 original images...
  ✓ Already have 4792 images (target: 2000)

📊 AUGMENTATION STATISTICS

Class                Original     Augmented    Total       
------------------------------------------------------------
Astrocytoma          1128         872          2000        
Ependymoma           300          1700         2000        
Glioblastoma         408          1592         2000        
Germinoma            200          1800         2000        
Medulloblastoma      262          1738         2000        
Meningioma           5164         0            5164        
Oligodendroglioma    448          1552         2000        
Pituitary            5316         0            5316        
Schwannoma           930          1070         2000        
No_Tumor             4792         0            4792        

Statistics saved to: brain_mri_10class_augmented\augmentation_stats.json

✅ Augmentation completed!
📂 Augmented data saved to: brain_mri_10cla

In [6]:
import os

base_folder_path = 'E:/Semisters/brain tumer prediction/brain_mri_10class_augmented'

# List of subdirectories (your 'classes')
subdirectories = [
    'No_Tumor',
    'Medulloblastoma',
    'Germinoma',
    'Oligodendroglioma',
    'Astrocytoma',
    'Pituitary',
    'Meningioma',
    'Ependymoma',
    'Glioblastoma',
    'Schwannoma'
]

image_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp') # Common image file extensions

print(f"Counting files in subdirectories of: {base_folder_path}\n")

for sub_dir_name in subdirectories:
    sub_dir_path = os.path.join(base_folder_path, sub_dir_name)

    if os.path.exists(sub_dir_path) and os.path.isdir(sub_dir_path):
        file_count = 0
        for item in os.listdir(sub_dir_path):
            if os.path.isfile(os.path.join(sub_dir_path, item)) and item.lower().endswith(image_extensions):
                file_count += 1
        print(f"  '{sub_dir_name}': {file_count} image files")
    else:
        print(f"  Warning: Subdirectory '{sub_dir_name}' not found or is not a directory.")


Counting files in subdirectories of: E:/Semisters/brain tumer prediction/brain_mri_10class_augmented

  'No_Tumor': 2396 image files
  'Medulloblastoma': 1869 image files
  'Germinoma': 1900 image files
  'Oligodendroglioma': 1776 image files
  'Astrocytoma': 1436 image files
  'Pituitary': 2658 image files
  'Meningioma': 2582 image files
  'Ependymoma': 1850 image files
  'Glioblastoma': 1796 image files
  'Schwannoma': 1535 image files
